In [27]:
import importlib
import random
import numpy as np
from numpy import polyfit, poly1d
import argparse
from pathlib import Path
import shutil 
import os 
import sys
import re
import yaml
import gen_structures
from scipy.linalg import lapack
importlib.reload(gen_structures)
from gen_structures import mol_freq
from prettytable import PrettyTable

def Sort_EV(E,V):
    """
    sort the 1-d rate_list E, and get the corresponding eigenvector V
    """
    Elength = np.shape(E)[0]
    while Elength > 0:
        for i in range(0,Elength-1):
            if E[i].real > E[i+1].real:
                E[i], E[i+1] = E[i+1], E[i]
                V[:,[i,i+1]]=V[:,[i+1,i]]
        Elength -= 1
    return E,V

def get_Mod_square(a):
    """
    get Mod_square of a complex a 
    """
    return a.real**2+a.imag**2

def Expansion(A,B,j):
        table = PrettyTable(["JM","expansion real","expansion image","percentage"])
        print("expansion of state "+str(j))
        for i in range(0,16):
            x = 2*i+1-16
            C = A[j,i]+B[j,i]*1j
            table.add_row(['|{y}/2>'.format(y=x),A[j,i],B[j,i]*1j,'%1.1f'%(get_Mod_square(C)*100)+"%"])
        print(table)
        return table

def iTri(i,j):
    if j>i:
        iTri = j*(j-1)/2+i
    else:
        iTri = i*(i-1)/2+j
    return iTri

def read_HCF(filename,n_states):
    pattern = re.compile(r"[+-]?\d+\.\d+|[+-]?\d+")   #determine the re pattern of number needed to be found 
    A = np.empty([n_states,n_states], dtype = float)
    B = np.empty([n_states,n_states], dtype = float)
    with open(filename,"r") as file:
        while True:
            line = file.readline()
            if not line:
                break
            if len(line.split()) == 13 and " ".join(line.split()[:]) == "Ab Initio Calculated Crystal-Field Splitting Matrix written in the basis of Pseudospin Eigenfunctions":
                for k in range(0, n_states//4):
                    for i in range(0,3):
                        line = file.readline()
                    for i in range(0,16):
                        line = file.readline()
                        temp = pattern.findall(line.split('|')[2].split()[0])
                        #print(temp[0])
                        A[i,4*k] = float(temp[0])
                        temp = pattern.findall(line.split('|')[2].split()[1])
                        B[i,4*k] = float(temp[0])
                        temp = pattern.findall(line.split('|')[3].split()[0])
                        A[i,4*k+1] = float(temp[0])
                        temp = pattern.findall(line.split('|')[3].split()[1])
                        B[i,4*k+1] = float(temp[0])
                        temp = pattern.findall(line.split('|')[4].split()[0])
                        A[i,4*k+2] = float(temp[0])
                        temp = pattern.findall(line.split('|')[4].split()[1])
                        B[i,4*k+2] = float(temp[0])
                        temp = pattern.findall(line.split('|')[5].split()[0])
                        A[i,4*k+3] = float(temp[0])
                        temp = pattern.findall(line.split('|')[5].split()[1])
                        B[i,4*k+3] = float(temp[0])
                    line = file.readline()
                if  n_states%2 != 0:
                    for i in range(0,3):
                        line = file.readline()
                    for i in range(0,16):
                        line = file.readline()
                        for j in range(0, n_states%2):
                            a = line.split("|")[j+1]
                            temp = pattern.findall(line.split('|')[2].split()[0])
                            A[j+ n_states//2,i] = float(temp[0])*10**float(temp[1])
                            temp = pattern.findall(line.split('|')[2].split()[1])
                            B[j+ n_states//2,i] = float(temp[0])*10**float(temp[1])
    file.close()
    return A,B


n_states = 16
for i in range(1,7):
    filename = "inpfile/ro/eq_expx{x}.log".format(x=i)
    A,B = read_HCF(filename,n_states)
    C = A+B*1j
    EV =lapack.zheev(C)
    V = EV[1]
    V = V.T
    A = V.real
    B = V.imag
    file = open('Expan_x{x}.txt'.format(x=i),'w')
    for i in range(0,16):
        table=Expansion(A,B,i)
        print(table,file=file)



#print(C)
n = 16*(16+1)/2
ap = np.empty([int(n)], dtype = complex)
# for i in range(0,16):
#     for j in range(0,i):
#         ap[iTri(j,i)] = C[j,i]



expansion of state 0
+---------+-------------------------+------------------------------+------------+
|    JM   |      expansion real     |       expansion image        | percentage |
+---------+-------------------------+------------------------------+------------+
| |-15/2> |   7.14305617211014e-23  | (-0-7.181211875828315e-24j)  |    0.0%    |
| |-13/2> |  1.0816054435763216e-08 |   2.2870039300209154e-08j    |    0.0%    |
| |-11/2> |  4.5673881274851896e-08 |   1.0423317074203021e-08j    |    0.0%    |
|  |-9/2> |  9.019021626580892e-08  | (-0-4.2653378459032055e-08j) |    0.0%    |
|  |-7/2> |  2.1856809922206207e-07 |    6.076447272890162e-08j    |    0.0%    |
|  |-5/2> |  5.614207475953237e-07  |    1.020286242188978e-06j    |    0.0%    |
|  |-3/2> |  8.787906842540726e-07  |    4.595281831944689e-07j    |    0.0%    |
|  |-1/2> | -1.0048983726605255e-05 | (-0-5.2278141835792245e-06j) |    0.0%    |
|  |1/2>  |  -8.123213899159877e-05 |  (-0-3.21226537093881e-06j)  |    0.0% 

In [4]:
import numpy as np
from numpy import polyfit, poly1d
import argparse
from pathlib import Path
import shutil 
import os 
import sys
import re

import ase as ase
from ase import Atoms
from ase import Atom
from ase.io import read
from ase.visualize import view
from ase.build import fcc111, add_adsorbate
from ase.io import write
from ase.io import vasp

symbols = {
    1: "H",    2: "He",    
    3: "Li",    4: "Be",    5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",    
    11: "Na",    12: "Mg",    13: "Al",    14: "Si",    15: "P",    16: "S",    17: "Cl",    18: "Ar",    
    19: "K",    20: "Ca",    21: "Sc",    22: "Ti",    23: "V",    24: "Cr",    25: "Mn",    26: "Fe",    27: "Co",    28: "Ni",    29: "Cu",    30: "Zn",    31: "Ga",    32: "Ge",    33: "As",    34: "Se",    35: "Br",    36: "Kr",
    37: "Rb",    38: "Sr",    39: "Y",    40: "Zr",    41: "Nb",    42: "Mo",    43: "Tc",    44: "Ru",    45: "Rh",    46: "Pd",    47: "Ag",    48: "Cd",    49: "In",    50: "Sn",    51: "Sb",    52: "Te",    53: "I",    54: "Xe",    
    55: "Cs",   56: "Ba",    
    57: "La",    58: "Ce",    59: "Pr",    60: "Nd",    61: "Pm",    62: "Sm",    63: "Eu",    64: "Gd",    65: "Tb",    66: "Dy",    67: "Ho",    68: "Er",    69: "Tm",    70: "Yb",    
    71: "Lu",    72: "Hf",    73: "Ta",    74: "W",    75: "Re",    76: "Os",    77: "Ir",    78: "Pt",    79: "Au",    80: "Hg",    81: "Tl",    82: "Pb",    83: "Bi",    84: "Po",    
    85: "At",    86: "Rn",
}


element_dict = {
    "H": 1,    "He": 2,    
    "Li": 3,    "Be": 4,    "B": 5,    "C": 6,    "N": 7,    "O": 8,    "F": 9,    "Ne": 10,    
    "Na": 11,    "Mg": 12,    "Al": 13,    "Si": 14,    "P": 15,    "S": 16,    "Cl": 17,    "Ar": 18,    
    "K": 19,    "Ca": 20,    "Sc": 21,    "Ti": 22,    "V": 23,    "Cr": 24,    "Mn": 25,    "Fe": 26,    "Co": 27,    "Ni": 28,    "Cu": 29,    "Zn": 30,    "Ga": 31,    "Ge": 32,    "As": 33,    "Se": 34,    "Br": 35,    "Kr": 36,
    "Rb": 37,    "Sr": 38,    "Y": 39,    "Zr": 40,    "Nb": 41,    "Mo": 42,    "Tc": 43,    "Ru": 44,    "Rh": 45,    "Pd": 46,    "Ag": 47,    "Cd": 48,    "In": 49,    "Sn": 50,    "Sb": 51,    "Te": 52,    "I": 53,    "Xe": 54,    
    "Cs": 55,    "Ba": 56,    
    "La": 57,    "Ce": 58,    "Pr": 59,    "Nd": 60,    "Pm": 61,    "Sm": 62,    "Eu": 63,    "Gd": 64,    "Tb": 65,    "Dy": 66,    "Ho": 67,    "Er": 68,    "Tm": 69,    "Yb": 70,    
    "Lu": 71,    "Hf": 72,    "Ta": 73,    "W": 74,    "Re": 75,    "Os": 76,    "Ir": 77,    "Pt": 78,    "Au": 79,    "Hg": 80,    "Tl": 81,    "Pb": 82,    "Bi": 83,   
}


def Expand_group(orig,Select_Atnum,length,dpxso=None):
    """
    Use length as radius for each atom in the atom set Select_Atnum, select the atoms within the radius into the atom set, and output the new atom set Select_Atnum. 

    Args:
        orig (atoms): the target molecule.
        Select_Atnum (np): the original atom set.
        length (float): the radius of search
        dpxso (np): It store the imformation of the translation of the atoms. If input it,the function regard the molecule as a cell.

    Returns:
        Select_Atnum (np): the final atom set.
        dpxs
    """
    if dpxso is None:
        for j in Select_Atnum:
            for i in range(0,orig.get_number_of_atoms()):
                l0 = orig.get_positions()[j]
                l1 = orig.get_positions()[i]
                dl = np.sqrt(np.sum((l0-l1)**2))
                #print(i,orig.get_chemical_symbols()[i],l1,dl)
                if dl < length:
                    Select_Atnum = np.append(Select_Atnum,i)
                    #print(dl,length,Select_Atnum)
        Select_Atnum = np.unique(Select_Atnum)
        Select_Atnum = np.sort(Select_Atnum)
        print(Select_Atnum)
        return Select_Atnum
    else:
        dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
        for j in Select_Atnum:
            for i in range(0,orig.get_number_of_atoms()):
                cell = orig.get_cell()
                l0 = orig.get_positions()[j]
                l1 = orig.get_positions()[i]
                l1,dl,dpx = Select_l(l1,l0,cell)
                #dl = np.sqrt(np.sum((l0-l1)**2))
                if dl < length:
                    Select_Atnum = np.append(Select_Atnum,i)
                    dpxs[i,...] = dpx+dpxso[j,...]
                #if i == 1:
                    #print(dpxs[i,...])
                #dl = np.sqrt((l0[0]-l1[0])**2+(l0[1]-l1[1])**2+(l0[2]-l1[2])**2)
                #print(dl)

        Select_Atnum = np.unique(Select_Atnum)
        Select_Atnum = np.sort(Select_Atnum)
        print(Select_Atnum)
        return Select_Atnum,dpxs

def Select_l(l1,l0,cell):
    """
    Find the nearest distance of two position in the period condition

    Args:
        l0: the position not changed
        l1: the position will changed by the periodic lattice
        cell: the lattice vector

    Returns:
        lx: the nearsest position of l0
        dl: the nearest distance
        dpx: the change by the lattice vector
    """
    lx = l1
    dl = np.sqrt(np.sum((l0-l1)**2))
    dpx = np.zeros([3],dtype = int)
    for i in range(-1,2):
        for j in range(-1,2):
            for k in range(-1,2):
                l = l1+i*cell[0,...]+j*cell[1,...]+k*cell[2,...]
                dl2 = np.sqrt(np.sum((l0-l)**2))
                if dl2 < dl:
                    lx = l
                    dl = dl2
                    dpx = [i,j,k]
    return lx,dl,dpx

def check_occupy(orig):
    """
    Delete the atoms whose occupancy is lower than 0.5

    Args:
        orig (atoms): the target molecule.

    Returns:
        new (atoms): the final molecule.
    """
    n_cell = int(len(orig)/len(orig.info['occupancy']))
    Select_Atnum = np.zeros([(len(orig))],dtype = int)
    for i in range(len(orig)):
        Select_Atnum[i] = int(i)
    num = Select_Atnum[0]


    positions = tuple(orig.get_positions()[num])
    new = Atoms(orig.get_chemical_symbols()[Select_Atnum[0]],[positions],cell = orig.get_cell())#add the first atom in new
    
    
    j = 1
    for i in range(len(Select_Atnum)-1):
        num = Select_Atnum[i+1]        
        numdict = (i+1)//n_cell
        dit = orig.info['occupancy']["{x}".format(x=numdict)]
        if list(dit.values())[0]>0.5:
            #print(i,num,list(dit.values())[0])
            new.append(orig.get_chemical_symbols()[num])
            new.positions[j] = tuple(orig.get_positions()[num])
            j = j+1
    #print(xray.get_number_of_atoms())
    #view(xray)
    return new

def choose_group(orig,center_atom_name,outfold,r1,r2):
    """
    Select the set of atoms from the molecule using this way:
    (1)Choose a center atom
    (2)use Expand_group function with length=r1 to select the Select_Atnum once. 
    (3)use Expand_group function with length=r2 to select the Select_Atnum untill the amount of atoms unchanged or 20 times.
    (4)return the new atoms

    Args:
        orig (atoms): the target molecule.
        center_atom_name (str): the name of the center atom, such as 'Dy'.
        outfold (str): the path of the output fold
        r1 (float)
        r2 (float)

    Returns:
        new (atoms): the final molecule.
    """
    initial_num = np.where(orig.numbers == element_dict[center_atom_name])[0][0]
    Select_Atnum = np.asarray([initial_num])
    cell = orig.get_cell()

    dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
    #以r1为半径进行第一次选择
    Select_Atnum,dpxs = Expand_group(orig,Select_Atnum,r1,dpxs)
    print("Start select group")

    #以r2为半径不断选择，指导原子集数量不变
    for i in range(0,20):
        num1 = len(Select_Atnum)
        Select_Atnum,dpxs = Expand_group(orig,Select_Atnum,r2,dpxs)
        num2 = len(Select_Atnum)
        if num1 == num2:
            break

    #输出目标原子序号Atom_list，晶格移动编号dpxs
    np.savetxt(outfold+"Atom_list",Select_Atnum)
    np.savetxt(outfold+"dpxs",dpxs)
    print("output dpxs")

    #"""
    """
    dpxs = np.loadtxt("dpxs",dtype = int)
    #print(dpxs)
    list1 = "3   7  11  15  19  23  27  31  35  39  43  47  51  55  59  63  75  87\
        91  95  99 103 111 115 119 123 127 135 139 143 147 159 175 207 211 215\
        219 223 235 239 243 247 251 255 259 263 275 279 283 287 291 319 355"
    li = list1.split()
    Select_Atnum = np.asarray(li,dtype=int)
    #print(Select_Atnum2)
    """
    #按照目标原子编号生成新的结构


    num = Select_Atnum[0]
    #读取初始位置
    positions = tuple(orig.get_positions()[num])+dpxs[num,0]*cell[0,...]+dpxs[num,1]*cell[1,...]+dpxs[num,2]*cell[2,...]
    new = Atoms(orig.get_chemical_symbols()[Select_Atnum[0]],[positions])

    #按照目标原子编号依次在new中添加原子
    for i in range(len(Select_Atnum)-1):
        num = Select_Atnum[i+1]
        new.append(orig.get_chemical_symbols()[num])
        new.positions[i+1] = tuple(orig.get_positions()[num])+dpxs[num,0]*cell[0,...]+dpxs[num,1]*cell[1,...]+dpxs[num,2]*cell[2,...]
    print(new.get_number_of_atoms())

    #输出结构
    #view(new)
    return new

def choose_atoms(orig,center_atom_name,choose_atom_name):
    """
    Select the set of atoms including one center_atom and all choose_atom_name atoms.


    Args:
        orig (atoms): the target molecule.
        center_atom_name (str): the name of the center atom, such as 'Dy'.
        choose_atom_name (str): the name of the choosed atom, such as 'C'.

    Returns:
        new (atoms): the final molecule.
    """
    initial_num = np.where(orig.numbers == element_dict[center_atom_name])[0][0]
    atom_distance_list = np.zeros([orig.get_number_of_atoms()],dtype = float)

    for i in range(0,orig.get_number_of_atoms()):
        if orig.get_chemical_symbols()[i] == choose_atom_name:
            l0 = orig.get_positions()[0]
            l1 = orig.get_positions()[i]
            dl = np.sqrt(np.sum((l0-l1)**2))
            atom_distance_list[i] = dl

    little_list = np.argsort(atom_distance_list)
    positions = tuple(orig.get_positions()[initial_num])
    new = Atoms(orig.get_chemical_symbols()[initial_num],[positions])
    j = 0
    for i in range(little_list.shape[0]):
        if j >= 10:
            break
        if atom_distance_list[little_list[i]]>0:
            num = little_list[i]
            new.append(orig.get_chemical_symbols()[num])
            new.positions[j+1] = tuple(orig.get_positions()[num])
            j = j+1

    return new

def addH_to_Cp(orig,r1,rH):
    """
    Add H atoms to the atoms with Dy and tow Cp rings.

    Args:
        orig (atoms): the molecule has one Dy atom and tow Cprings.
        r1 (float): the value, which is larger than the side length of theCprings, but should be lower than the distance of Dy-C. 1.5A is a good choice.
        rH (float): the value of the distance between the C and H, 0.95A is a good choice.

    Returns:
        orig (atoms): the final molecule.
    """
    Cp_Atnum1 = np.zeros([1],dtype = int)
    Cp_Atnum2 = np.zeros([1],dtype = int)

    for i in range(0,orig.get_number_of_atoms()):
        if orig.get_chemical_symbols()[i] == "C":
            Cp_Atnum1[0] = i
            break

    for i in range(0,20):
        num1 = len(Cp_Atnum1)
        Cp_Atnum1 = Expand_group(orig,Cp_Atnum1,r1)
        num2 = len(Cp_Atnum1)
        if num1 == num2:
            break


    for i in range(0,orig.get_number_of_atoms()):
        if orig.get_chemical_symbols()[i] == "C":
            if not np.isin(i, Cp_Atnum1):
                print(i)
                Cp_Atnum2[0] = i
                break

    for i in range(0,20):
        num1 = len(Cp_Atnum2)
        Cp_Atnum2 = Expand_group(orig,Cp_Atnum2,r1)
        num2 = len(Cp_Atnum2)
        if num1 == num2:
            break

    center1 = np.zeros([3],dtype = float)
    for j in Cp_Atnum1:
        l0 = orig.get_positions()[j]
        center1 = center1+l0/Cp_Atnum1.shape
    for j in Cp_Atnum1:
        vector = orig.get_positions()[j]-center1
        vector = vector/np.sqrt(np.sum(vector**2))
        posH = orig.get_positions()[j]+vector*rH
        orig.append(Atom('H',posH))

    center2 = np.zeros([3],dtype = float)
    for j in Cp_Atnum2:
        l0 = orig.get_positions()[j]
        center2 = center2+l0/Cp_Atnum2.shape
    for j in Cp_Atnum2:
        vector = orig.get_positions()[j]-center2
        vector = vector/np.sqrt(np.sum(vector**2))
        posH = orig.get_positions()[j]+vector*rH
        orig.append(Atom('H',posH))
    return orig
    




if __name__ == "__main__":
    """parser = argparse.ArgumentParser(description=""
        A script to select groups from Crystal structure."")
    parser.add_argument("--cry", dest="cryfile",type=str, required=True, help="The filename of Crystal structure.")
    parser.add_argument("--ct", dest="center", type=str, default="None", help="Element of the center of Stru.")
    parser.add_argument("--num",  dest="number", type=int, help="The number of the center atom of Stru.")
    parser.add_argument("--out", dest="outfold",type=str, help="The fold where the out file output")
    parser.add_argument("--freq", "-f", action="store_true", help="output freq-POSCAR.")
    args = parser.parse_args()"""
    #选择目标原子集

    cryfile = "inpfile/POSCAR"
    outfold = "inpfile/"
    groupfold = outfold
    center_atom_name = "Dy"
    choose_atom_name = "C"
    r1 = 1.5
    rH = 0.95

    #读取晶体数据
    orig = read(cryfile)
    view(orig)
    print(orig.get_distance(0,178))
    #得出晶格参数
    cell = orig.get_cell()
    
    #确定初始原子集，默认为单个args.center命名的原子，一般为Co或Dy
    initial_num = np.where(orig.numbers == element_dict[center_atom_name])[0][0]
    Select_Atnum = np.asarray([initial_num])
    #print(Select_Atnum)
    
    
    dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
    #print(dpxs)
    group = choose_group(orig,center_atom_name,outfold,2.9,1.7)

    view(group)
    #write(outfold+"test.xyz",group)

12.26838830136422
[ 68 227 228 229 231 232 233 237 242]
Start select group


C:\Users\10122\AppData\Local\Temp\ipykernel_28588\106905421.py:364: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
C:\Users\10122\AppData\Local\Temp\ipykernel_28588\106905421.py:179: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
C:\Users\10122\AppData\Local\Temp\ipykernel_28588\106905421.py:73: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  dpxs = np.zeros([orig.get_number_of_atoms(),3],dtype = int)
C:\Users\10122\AppData\Local\Temp\ipykernel_28588\106905421.py:75: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  for i in range(0,orig.get_number_of_atoms()):


[ 18  20  23  28  30  34  68 193 198 202 203 227 228 229 231 232 233 237
 242]
[  2   5   8  18  19  20  22  23  25  26  28  29  30  33  34  68 193 198
 202 203 204 227 228 229 231 232 233 237 242]
[  2   5   8  18  19  20  22  23  25  26  28  29  30  33  34  51  68 100
 140 141 142 143 144 145 146 147 148 149 150 151 153 166 177 178 183 193
 195 197 198 199 200 202 203 204 227 228 229 231 232 233 237 242]
[  2   5   8  18  19  20  22  23  25  26  28  29  30  33  34  44  51  68
  98  99 100 140 141 142 143 144 145 146 147 148 149 150 151 153 166 177
 178 183 193 195 197 198 199 200 202 203 204 227 228 229 231 232 233 237
 242 243]
[  2   5   8  18  19  20  22  23  25  26  28  29  30  33  34  44  51  68
  97  98  99 100 101 102 109 140 141 142 143 144 145 146 147 148 149 150
 151 153 166 177 178 183 189 193 195 197 198 199 200 202 203 204 227 228
 229 231 232 233 237 242 243]
[  2   5   8  18  19  20  22  23  25  26  28  29  30  33  34  44  51  68
  97  98  99 100 101 102 109 140 141 14

C:\Users\10122\AppData\Local\Temp\ipykernel_28588\106905421.py:221: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  print(new.get_number_of_atoms())
